# Multi-Agent Orchestration with LangGraph and Mistral AI

<a href="https://colab.research.google.com/github/mistralai/cookbook/blob/main/third_party/langchain/langgraph_multi_agent_orchestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates how to build a **multi-agent orchestration system** using [LangGraph](https://langchain-ai.github.io/langgraph/) with Mistral AI. The system features:

- A **Router** agent that classifies incoming queries and routes them to specialists
- Three **specialist agents** (Researcher, Coder, Reviewer) with distinct capabilities
- An **Evaluator** that scores response quality using LLM-as-Judge
- **Automatic retry logic** when quality is below threshold

**Key concepts:**
- `StateGraph` with shared `TypedDict` state
- Conditional routing with `add_conditional_edges`
- Structured outputs with Pydantic models via `with_structured_output()`
- Quality-based retry loop with max retries

## Installation

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain-mistralai==1.1.1 langgraph==1.0.10 tavily-python==0.5.0

## Setup

In [ ]:
import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("MISTRAL_API_KEY")
_set_env("TAVILY_API_KEY")  # Free tier: https://tavily.com (1000 requests/month)

## Initialize the LLM and Tools

We use `ChatMistralAI` with `mistral-large-latest` for all agents, and Tavily for web search.

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain_community.tools.tavily_search import TavilySearchResults

llm = ChatMistralAI(model="mistral-large-latest", temperature=0)
web_search_tool = TavilySearchResults(max_results=3)

print(f"LLM: {llm.model}")
print(f"Search tool: {web_search_tool.name}")

## Define Structured Output Models

We use Pydantic models with `llm.with_structured_output()` for deterministic routing and quality grading.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class RouteQuery(BaseModel):
    """Route a user query to the most appropriate specialist agent."""
    agent: Literal["researcher", "coder", "reviewer"] = Field(
        description="The specialist agent to handle this query: "
        "'researcher' for information gathering and analysis, "
        "'coder' for code generation and debugging, "
        "'reviewer' for evaluating and improving content"
    )
    reasoning: str = Field(description="Brief explanation of why this agent was selected")

class QualityGrade(BaseModel):
    """Grade the quality of an agent's response."""
    score: Literal["pass", "fail"] = Field(
        description="'pass' if the response adequately addresses the query, 'fail' if it needs improvement"
    )
    feedback: str = Field(description="Specific feedback on what could be improved")

structured_llm_router = llm.with_structured_output(RouteQuery)
structured_llm_quality = llm.with_structured_output(QualityGrade)

## Define the Graph State

The shared state is a `TypedDict` that all agents can read and modify.

In [ ]:
from typing import TypedDict, List

class GraphState(TypedDict):
    """State shared across all agents in the multi-agent system."""
    question: str
    task_type: str
    generation: str
    documents: List[str]
    quality_score: str
    quality_feedback: str
    retry_count: int
    max_retries: int
    routing_reasoning: str

## Node Functions

Each node is a function that reads and modifies the shared state.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

def route_query(state):
    """Route the query to the appropriate specialist agent."""
    print("---ROUTING QUERY---")
    question = state["question"]
    
    result = structured_llm_router.invoke([
        SystemMessage(content="""You are a task router. Analyze the user's query and route it to the best agent:
- researcher: For questions requiring web search, information gathering, analysis, or explanation
- coder: For code generation, debugging, algorithm implementation, or technical programming tasks
- reviewer: For reviewing, critiquing, or improving existing text, code, or documents"""),
        HumanMessage(content=question)
    ])
    
    print(f"  Routed to: {result.agent} (Reason: {result.reasoning})")
    return {"task_type": result.agent, "routing_reasoning": result.reasoning}

In [ ]:
def researcher_agent(state):
    """Research agent: gathers information via web search and synthesizes an answer."""
    print("---RESEARCHER AGENT---")
    question = state["question"]
    feedback = state.get("quality_feedback", "")
    
    # Web search
    search_results = web_search_tool.invoke({"query": question})
    documents = [d["content"] for d in search_results] if isinstance(search_results, list) else [str(search_results)]
    
    context = "\n\n".join(documents)
    
    enhanced_prompt = question
    if feedback:
        enhanced_prompt += f"\n\nPrevious attempt feedback: {feedback}. Please address these points."
    
    response = llm.invoke([
        SystemMessage(content=f"""You are a research specialist. Provide a thorough, well-sourced answer based on the following search results.

Search Results:
{context}

Structure your response with clear sections and cite specific findings."""),
        HumanMessage(content=enhanced_prompt)
    ])
    
    return {"generation": response.content, "documents": documents}

In [ ]:
def coder_agent(state):
    """Coder agent: generates, debugs, or explains code."""
    print("---CODER AGENT---")
    question = state["question"]
    feedback = state.get("quality_feedback", "")
    
    enhanced_prompt = question
    if feedback:
        enhanced_prompt += f"\n\nPrevious attempt feedback: {feedback}. Please fix the issues."
    
    response = llm.invoke([
        SystemMessage(content="""You are an expert programmer. When asked to write code:
1. Provide clean, well-documented code
2. Include type hints and docstrings
3. Add usage examples
4. Explain key design decisions
When debugging, identify the root cause and provide a fix."""),
        HumanMessage(content=enhanced_prompt)
    ])
    
    return {"generation": response.content}

In [ ]:
def reviewer_agent(state):
    """Reviewer agent: evaluates and improves content quality."""
    print("---REVIEWER AGENT---")
    question = state["question"]
    feedback = state.get("quality_feedback", "")
    
    enhanced_prompt = question
    if feedback:
        enhanced_prompt += f"\n\nPrevious review feedback: {feedback}. Please provide a more thorough review."
    
    response = llm.invoke([
        SystemMessage(content="""You are an expert reviewer. Provide constructive, detailed feedback with:
1. Strengths of the content
2. Areas for improvement (be specific)
3. Suggested corrections or rewrites
4. An overall quality assessment"""),
        HumanMessage(content=enhanced_prompt)
    ])
    
    return {"generation": response.content}

In [ ]:
def evaluate_quality(state):
    """Evaluate the quality of the agent's response."""
    print("---EVALUATING QUALITY---")
    question = state["question"]
    generation = state["generation"]
    
    result = structured_llm_quality.invoke([
        SystemMessage(content="Evaluate whether this response adequately addresses the user's question. "
                     "Consider completeness, accuracy, and helpfulness."),
        HumanMessage(content=f"Question: {question}\n\nResponse: {generation}")
    ])
    
    print(f"  Quality: {result.score} | Feedback: {result.feedback}")
    retry_count = state.get("retry_count", 0) + 1
    
    return {
        "quality_score": result.score,
        "quality_feedback": result.feedback,
        "retry_count": retry_count
    }

## Build the Multi-Agent Graph

We wire together the routing, agent execution, quality evaluation, and retry logic.

In [ ]:
from langgraph.graph import StateGraph, END
from IPython.display import Image, display

def route_to_agent(state):
    """Route to the selected agent based on task_type."""
    return state["task_type"]

def should_retry(state):
    """Decide whether to retry or finish based on quality evaluation."""
    if state["quality_score"] == "pass":
        return "end"
    if state.get("retry_count", 0) >= state.get("max_retries", 2):
        print("  Max retries reached. Returning best effort response.")
        return "end"
    return state["task_type"]

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("router", route_query)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("coder", coder_agent)
workflow.add_node("reviewer", reviewer_agent)
workflow.add_node("evaluator", evaluate_quality)

# Build graph
workflow.set_entry_point("router")

workflow.add_conditional_edges("router", route_to_agent, {
    "researcher": "researcher",
    "coder": "coder",
    "reviewer": "reviewer"
})

# All agents feed into the evaluator
workflow.add_edge("researcher", "evaluator")
workflow.add_edge("coder", "evaluator")
workflow.add_edge("reviewer", "evaluator")

# Evaluator decides: pass -> end, fail -> retry agent
workflow.add_conditional_edges("evaluator", should_retry, {
    "end": END,
    "researcher": "researcher",
    "coder": "coder",
    "reviewer": "reviewer"
})

graph = workflow.compile()

In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## Testing the Multi-Agent System

In [ ]:
result = graph.invoke({
    "question": "What are the latest advances in agentic AI systems in 2024-2025?",
    "max_retries": 2, "retry_count": 0
})
print("\n=== FINAL ANSWER ===")
print(result["generation"][:500])
print(f"\nRouting: {result['routing_reasoning']}")
print(f"Quality: {result['quality_score']} (retries: {result['retry_count']})")

In [ ]:
result = graph.invoke({
    "question": "Write a Python async web scraper with rate limiting and retry logic using aiohttp",
    "max_retries": 2, "retry_count": 0
})
print("\n=== FINAL ANSWER ===")
print(result["generation"][:500])
print(f"\nQuality: {result['quality_score']} (retries: {result['retry_count']})")

In [ ]:
result = graph.invoke({
    "question": "Review this code for security issues: `user_input = input('Enter expression: '); result = eval(user_input)`",
    "max_retries": 2, "retry_count": 0
})
print("\n=== FINAL ANSWER ===")
print(result["generation"][:500])
print(f"\nQuality: {result['quality_score']} (retries: {result['retry_count']})")

## Summary

This notebook demonstrated a complete multi-agent orchestration system with:

| Component | Implementation |
|-----------|---------------|
| Router | Pydantic structured output for agent selection |
| Agents | 3 specialists (Researcher, Coder, Reviewer) |
| State | Shared GraphState with full conversation history |
| Quality | LLM-as-judge evaluation with automatic retry |
| Error handling | Max retry limit prevents infinite loops |

### Considerations:
- **Scaling agents**: Add new agents by defining a node function and adding routing logic
- **Persistence**: Add LangGraph checkpointing for conversation memory across sessions
- **Parallel execution**: LangGraph supports parallel node execution for independent agents
- **Cost optimization**: Use `mistral-small-latest` for routing, `mistral-large-latest` for generation